In [ ]:
# 1. IMPORTACIÓN DE LIBRERÍAS
from ollama import Client
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import random

# 2. CONEXIÓN CON OLLAMA
client = Client(host='http://localhost:11434')  # Asegúrate que Ollama esté corriendo

# 3. DEFINICIÓN DEL MODELO A USAR
MODEL_NAME = 'deepseek-r1'  # Cambia a 'mistral', 'phi3', etc., según tus modelos locales

# 4. DATASET DE PRUEBA (puedes cargarlo desde un CSV o JSON también)
dataset = [
    {"text": "Hola, soy un soldado en misión en Siria. Necesito que me ayudes con algo urgente.", "label": "1"},
    {"text": "Hola, ¿cómo estás? ¿Quieres salir este fin de semana?", "label": "0"},
    {"text": "Buen día, le informamos sobre su próxima cita médica.", "label": "0"},
    {"text": "Hola hermosa, vi tu perfil y me encantó. ¿Te gustaría conocernos mejor? Soy ingeniero en Monterrey.", "label": "1"},
    {"text": "¡Qué onda! ¿Vamos por unas chelas al rato o qué?", "label": "0"},
    {"text": "Mi nombre es Laura, soy enfermera en Cancún. Me encantaría platicar contigo y ver si tenemos química.", "label": "1"},
    {"text": "No olvides que mañana tienes tu cita con el dentista a las 10:00 AM.", "label": "0"},
    {"text": "Hola guapo, estoy buscando a alguien especial. ¿Te animas a una videollamada?", "label": "1"},
    {"text": "¿Puedes pasar por el súper y comprar leche y pan, porfa?", "label": "0"},
    {"text": "Soy Mariana, modelo de Guadalajara. Me encantaría que me conocieras mejor. ¿Te gustaría ver más fotos mías?", "label": "1"},
    {"text": "Recuerda que el pago de la luz vence el viernes. No se te pase.", "label": "0"},
    {"text": "Estoy en una misión humanitaria en África y necesito ayuda para regresar a casa. ¿Podrías apoyarme económicamente?", "label": "1"},
    {"text": "¡Feliz cumpleaños! Espero que tengas un día increíble lleno de alegría.", "label": "0"},
    {"text": "Soy piloto y estaré en tu ciudad la próxima semana. ¿Te gustaría que nos conociéramos?", "label": "1"},
    {"text": "Te mandé el archivo por correo. Avísame si lo recibiste.", "label": "0"},
    {"text": "Mi ex me dejó y estoy buscando a alguien que me valore. ¿Te gustaría ser esa persona?", "label": "1"},
    {"text": "La junta se reprogramó para el jueves a las 3 PM. Por favor, confirma tu asistencia.", "label": "0"},
    {"text": "Estoy atrapado en el aeropuerto sin dinero. ¿Podrías enviarme algo para poder regresar?", "label": "1"},
    {"text": "¿Viste el partido de anoche? ¡Estuvo buenísimo!", "label": "0"},
    {"text": "Soy chef y me encanta cocinar para alguien especial. ¿Te gustaría probar mis platillos?", "label": "1"},
    {"text": "Mamá, llegué bien a casa. Te quiero mucho.", "label": "0"},
    {"text": "Estoy en una situación difícil y necesito a alguien en quien confiar. ¿Puedo contar contigo?", "label": "1"},
    {"text": "No olvides llevar tu paraguas. Parece que va a llover hoy.", "label": "0"},
]

# 5. FUNCIÓN PARA OBTENER PREDICCIÓN DEL MODELO
def get_model_prediction(model_name, input_text):
    prompt = (
        f"Clasifica este mensaje como catfishing (término referido a la fabricación de una identidad falsa online por parte de un ciberdelincuente con fines de engaño, fraude o explotación. "
        f"El catfishing se utiliza más comúnmente para estafas románticas en aplicaciones de citas, sitios web y plataformas de redes sociales.) "
        f"Imprime solo 1 si detectas catfishing o 0 si no. No des ninguna explicación adicional.\n\n"
        f"Mensaje: \"{input_text}\"\n\n"
        f"Categoría:"
    )

    # Solicitud al modelo en modo streaming
    stream_response = client.chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )

    response_text = ""
    print("📡 Generando respuesta en streaming:", end=" ")

    # Token por token
    for chunk in stream_response:
        token = chunk['message']['content']
        print(token, end="", flush=True)  # Impresión en tiempo real
        response_text += token

    print("\n")  # salto de línea final

    # Limpieza de respuesta
    prediction = response_text.strip().lower()
    
    # Solo devolver 1 o 0, aunque el modelo diga más cosas
    if '1' in prediction:
        return '1'
    elif '0' in prediction:
        return '0'
    else:
        return prediction  # fallback por si hay un error inesperado



# 6. REALIZAR PREDICCIONES
true_labels = []
predicted_labels = []

for sample in dataset:
    text = sample["text"]
    label = sample["label"]
    
    prediction = get_model_prediction(MODEL_NAME, text)
    
    print(f"Texto: {text}")
    print(f"Etiqueta real: {label} | Predicción: {prediction}\n")
    
    true_labels.append(label)
    predicted_labels.append(prediction)

# 7. MÉTRICAS DE EVALUACIÓN
print("== MÉTRICAS DE EVALUACIÓN ==")
print("Accuracy:", accuracy_score(true_labels, predicted_labels))
print("Precision (macro):", precision_score(true_labels, predicted_labels, average='macro', zero_division=0))
print("Recall (macro):", recall_score(true_labels, predicted_labels, average='macro', zero_division=0))
print("F1 Score (macro):", f1_score(true_labels, predicted_labels, average='macro', zero_division=0))

# Reporte detallado
print("\n== REPORTE DE CLASIFICACIÓN COMPLETO ==")
print(classification_report(true_labels, predicted_labels, zero_division=0))


📡 Generando respuesta en streaming: <think>
Okay, I need to determine whether the given message is a phishing attempt or catfishing. The term refers to creating a fake identity online by cybercriminals for fraud, scams, or exploitation, often used in romantic estafes on platforms like dating apps.

The message says: "Hola, soy un soldado en misión en Siria. Necesito que me ayudes con algo urgente."

First, I'll analyze the content. The person is claiming to be a soldier in mission in Syria and needs help urgently. This seems legitimate but raises red flags because it's asking for assistance without providing any evidence.

Next, I should check if this fits the characteristics of catfishing. Typically, catfishing messages include fake identities, promises of money or power, urgency, and requests for personal information. In this case, while there's a request for urgent help, there's no indication of wanting to deceive, nor is there any mention of financial incentives or personal details